In [2]:
# ============================================================
# GASF 512x512 — Siamese FSL training
# ============================================================

from pathlib import Path
import sys

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image


# ------------------------------------------------------------
# Project root
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "data").is_dir():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise RuntimeError("Could not locate project root.")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models import Siamese2D
from src.losses import contrastive_loss


DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

CLASS_NAMES = ["normal", "irf", "orf"]

IMAGE_ROOT = (
    PROJECT_ROOT
    / "datasets"
    / "images_512"
)

MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)

print("Image root:", IMAGE_ROOT)

Device: cpu
Image root: c:\Users\asus\OneDrive\Desktop\siamese-network\datasets\images_512


In [3]:
# ============================================================
# Load fixed FSL split
# ============================================================

split = np.load(
    PROJECT_ROOT
    / "results"
    / "fsl_split.npz"
)

train_idx = split["train_idx"]
val_idx = split["val_idx"]
test_idx = split["test_idx"]

y = split["y"]

print("Train:", len(train_idx))
print("Validation:", len(val_idx))
print("Test:", len(test_idx))

print(
    "Train class counts:",
    np.bincount(y[train_idx])
)

print(
    "Validation class counts:",
    np.bincount(y[val_idx])
)

print(
    "Test class counts:",
    np.bincount(y[test_idx])
)

Train: 210
Validation: 45
Test: 45
Train class counts: [70 70 70]
Validation class counts: [15 15 15]
Test class counts: [15 15 15]


In [4]:
# ============================================================
# Siamese pair dataset
# ============================================================

class SiameseImagePairDataset(Dataset):

    def __init__(
        self,
        indices,
        labels,
        image_root,
        representation,
        pairs_per_epoch=4000,
        seed=42
    ):
        self.indices = np.asarray(indices)
        self.labels = labels
        self.image_root = Path(image_root)
        self.representation = representation
        self.pairs_per_epoch = pairs_per_epoch
        self.rng = np.random.default_rng(seed)

        self.class_to_indices = {}

        for class_id in np.unique(labels):

            class_indices = self.indices[
                labels[self.indices] == class_id
            ]

            self.class_to_indices[
                int(class_id)
            ] = class_indices

    def __len__(self):
        return self.pairs_per_epoch

    def image_path(self, index):

        class_id = int(self.labels[index])
        class_name = CLASS_NAMES[class_id]

        local_index = int(index) % 100

        return (
            self.image_root
            / self.representation
            / class_name
            / f"{class_name}_{local_index:03d}.png"
        )

    def load_image(self, index):

        path = self.image_path(index)

        image = Image.open(path).convert("L")

        image = np.asarray(
            image,
            dtype=np.float32
        ) / 255.0

        image = torch.from_numpy(
            image
        ).unsqueeze(0)

        return image

    def __getitem__(self, idx):

        # 50% positive, 50% negative
        same_class = (
            self.rng.random() < 0.5
        )

        if same_class:

            class_id = int(
                self.rng.choice(
                    list(self.class_to_indices.keys())
                )
            )

            candidates = (
                self.class_to_indices[class_id]
            )

            i1, i2 = self.rng.choice(
                candidates,
                size=2,
                replace=False
            )

            label = 1.0

        else:

            classes = list(
                self.class_to_indices.keys()
            )

            class1, class2 = self.rng.choice(
                classes,
                size=2,
                replace=False
            )

            i1 = self.rng.choice(
                self.class_to_indices[class1]
            )

            i2 = self.rng.choice(
                self.class_to_indices[class2]
            )

            label = 0.0

        image1 = self.load_image(i1)
        image2 = self.load_image(i2)

        return (
            image1,
            image2,
            torch.tensor(
                label,
                dtype=torch.float32
            )
        )

In [5]:
# ============================================================
# DataLoaders
# ============================================================

BATCH_SIZE = 8

train_dataset = SiameseImagePairDataset(
    indices=train_idx,
    labels=y,
    image_root=IMAGE_ROOT,
    representation="gasf",
    pairs_per_epoch=4000,
    seed=42
)

val_dataset = SiameseImagePairDataset(
    indices=val_idx,
    labels=y,
    image_root=IMAGE_ROOT,
    representation="gasf",
    pairs_per_epoch=1000,
    seed=123
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print(
    "Train pairs:",
    len(train_dataset)
)

print(
    "Validation pairs:",
    len(val_dataset)
)

Train pairs: 4000
Validation pairs: 1000


In [6]:
# ============================================================
# Model
# ============================================================

model = Siamese2D(
    embedding_dim=128
).to(DEVICE)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

print(model)

Siamese2D(
  (encoder): Conv2DEncoder(
    (features): Sequential(
      (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): ReLU()
      (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (7): ReLU()
      (8): AdaptiveAvgPool2d(output_size=(1, 1))
    )
    (embedding): Linear(in_features=128, out_features=128, bias=True)
  )
)


In [ ]:
# ============================================================
# Train Siamese GASF 512x512
# ============================================================

NUM_EPOCHS = 20
MARGIN = 1.0

best_val_loss = float("inf")

checkpoint_path = (
    MODEL_DIR
    / "fsl_gasf_512_best.pt"
)


def run_epoch(
    model,
    loader,
    optimizer=None
):

    training = optimizer is not None

    if training:
        model.train()
    else:
        model.eval()

    losses = []

    for image1, image2, labels in loader:

        image1 = image1.to(DEVICE)
        image2 = image2.to(DEVICE)
        labels = labels.to(DEVICE)

        if training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training):

            z1, z2 = model(
                image1,
                image2
            )

            loss = contrastive_loss(
                z1,
                z2,
                labels,
                margin=MARGIN
            )

            if training:
                loss.backward()
                optimizer.step()

        losses.append(
            loss.item()
        )

    return float(
        np.mean(losses)
    )


for epoch in range(1, NUM_EPOCHS + 1):

    train_loss = run_epoch(
        model,
        train_loader,
        optimizer
    )

    val_loss = run_epoch(
        model,
        val_loader,
        optimizer=None
    )

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"Train loss: {train_loss:.5f} | "
        f"Val loss: {val_loss:.5f}"
    )

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "epoch": epoch,
                "val_loss": val_loss,
                "representation": "GASF",
                "image_size": 512,
                "embedding_dim": 128
            },
            checkpoint_path
        )

        print(
            "  ✓ Best checkpoint saved"
        )

print("\nTraining complete.")
print("Best checkpoint:")
print(checkpoint_path)

Epoch 01/20 | Train loss: 0.41230 | Val loss: 0.39312
  ✓ Best checkpoint saved
Epoch 02/20 | Train loss: 0.38730 | Val loss: 0.41506
Epoch 03/20 | Train loss: 0.37439 | Val loss: 0.46547
Epoch 04/20 | Train loss: 0.37118 | Val loss: 0.38904
  ✓ Best checkpoint saved
Epoch 05/20 | Train loss: 0.36665 | Val loss: 0.38397
  ✓ Best checkpoint saved
Epoch 06/20 | Train loss: 0.35768 | Val loss: 0.37112
  ✓ Best checkpoint saved
Epoch 07/20 | Train loss: 0.36587 | Val loss: 0.36626
  ✓ Best checkpoint saved
Epoch 08/20 | Train loss: 0.36168 | Val loss: 0.38716
Epoch 09/20 | Train loss: 0.35232 | Val loss: 0.36052
  ✓ Best checkpoint saved
Epoch 10/20 | Train loss: 0.35031 | Val loss: 0.39215
Epoch 11/20 | Train loss: 0.35426 | Val loss: 0.38479
Epoch 12/20 | Train loss: 0.35509 | Val loss: 0.39946
Epoch 13/20 | Train loss: 0.36042 | Val loss: 0.37817
Epoch 14/20 | Train loss: 0.34649 | Val loss: 0.35456
  ✓ Best checkpoint saved
Epoch 15/20 | Train loss: 0.34827 | Val loss: 0.36978
Epoch 16

: 